# Transfer learning on EuroSAT with ResNet-18 (Google Colab)

Fine-tunes an ImageNet-pretrained ResNet-18 on 27,000 Sentinel-2 satellite tiles (10 land-use classes) and saves `models/resnet18_eurosat.pt` for the Streamlit app.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Five epochs take roughly 10 minutes.

Replace `Ifychukwu` below with your GitHub username.

In [ ]:
!git clone https://github.com/Ifychukwu/robust-image-classification.git
%cd robust-image-classification
!pip -q install -r requirements-gpu.txt
import sys; sys.path.insert(0, "src")

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch the runtime to GPU)")

## 1. Look at the data

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets
ds = datasets.EuroSAT(root="data", download=True)
print(len(ds), "images,", len(ds.classes), "classes:", ds.classes)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for c, a in enumerate(axes.ravel()):
    i = next(j for j, (_, y) in enumerate(ds.samples) if y == c)
    a.imshow(ds[i][0]); a.set_title(ds.classes[c], fontsize=9); a.axis("off")
plt.tight_layout()

In [ ]:
import collections, pandas as pd
pd.Series(collections.Counter(ds.classes[y] for _, y in ds.samples)).sort_values().plot.barh(figsize=(7, 4), title="Images per class");

## 2. Fine-tune

AdamW, OneCycle learning rate, label smoothing 0.05, mixed precision, flips + rotations + colour jitter. The best epoch on validation is kept.

In [ ]:
from vision.transfer_learning import main
results = main(data="eurosat", epochs=5, batch_size=128)
pd.DataFrame(results["history"])

## 3. Test metrics

In [ ]:
print({k: v for k, v in results["test"].items() if k != "per_class"})
pc = pd.DataFrame(results["test"]["per_class"]).T.loc[results["classes"]]
pc[["precision", "recall", "f1-score", "support"]].sort_values("f1-score")

## 4. Grad-CAM: where did the network look?

In [ ]:
import numpy as np
from vision.transfer_learning import build_model, build_transforms, grad_cam
ckpt = torch.load("models/resnet18_eurosat.pt", map_location="cpu")
net = build_model(len(ckpt["classes"])); net.load_state_dict(ckpt["state_dict"])
tf = build_transforms()[1]
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for k, idx in enumerate(np.random.default_rng(0).choice(len(ds), 5, replace=False)):
    img, y = ds[int(idx)]
    t = tf(img.convert("RGB"))
    cam = grad_cam(net, t)
    pred = int(net.eval()(t[None]).argmax())
    axes[0, k].imshow(img); axes[0, k].set_title(f"true {ds.classes[y]}\npred {ds.classes[pred]}", fontsize=8)
    axes[1, k].imshow(img); axes[1, k].imshow(np.kron(cam, np.ones((64 // cam.shape[0],) * 2)), alpha=0.5, cmap="jet")
for a in axes.ravel(): a.axis("off")
plt.tight_layout()

## 5. Download the weights

Download `models/resnet18_eurosat.pt` and `reports/transfer_metrics_eurosat.json`, put them in the same folders of your local repo, then run `streamlit run app.py`. Add the test accuracy to the README results table.

In [ ]:
from google.colab import files
files.download("models/resnet18_eurosat.pt")
files.download("reports/transfer_metrics_eurosat.json")